In [32]:
from pyspark.sql import SparkSession
import time

In [54]:
spark = SparkSession.builder \
    .appName("Apex Financial Data Ingestion") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

print("🚀Spark UI Web URL:")
print(spark.sparkContext.uiWebUrl)

🚀Spark UI Web URL:
http://DESKTOP-H55HSBD.mshome.net:4041


In [ ]:
path = "c:/Users/u/Desktop/Repositories/Apex_financial/data/"
fact_transactions = spark.read.parquet(f"{path}gold/fact_transactions.parquet")

fact_transactions.show(5)

In [35]:
transactions_100k = (
    fact_transactions
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
    .unionByName(fact_transactions)
)

In [ ]:
print("the count of transactions_100k is: ", transactions_100k.count())
print("the count of fact_transactions is: ", fact_transactions.count())


from pyspark.sql import functions as F
import time

rounds = [1,2,3]
partitions = [4, 6, 8, 10, 12, 14, 16]

for i in rounds:
    print(f"Round {i}:")
    for p in partitions:
        spark.conf.set("spark.sql.shuffle.partitions", str(p))

        start = time.time()

        result = (
            transactions_100k
            .groupBy("customer_id")
            .agg(
                F.count("transaction_id").alias("transaction_count"),
                F.sum("amount").alias("total_amount"),
                F.avg("amount").alias("avg_transaction_amount")
            )
        )

        result.count()  # action → actually executes the Spark job

        elapsed = time.time() - start

        print(f"{p} partitions: {elapsed:.3f} seconds")

In [37]:
transactions_100k.rdd.getNumPartitions()

10

In [38]:
repartitioned = transactions_100k.repartition(20)

print("Original:", transactions_100k.rdd.getNumPartitions())
print("Repartitioned:", repartitioned.rdd.getNumPartitions())

Original: 10
Repartitioned: 20


In [ ]:
repartitioned.explain("formatted")

In [40]:
coalesced = transactions_100k.coalesce(5)

print("Original:", transactions_100k.rdd.getNumPartitions())
print("Coalesced:", coalesced.rdd.getNumPartitions())

Original: 10
Coalesced: 5


In [ ]:
coalesced.explain("formatted")

# experinmenting Broadcast joins

In [49]:
from pyspark.sql import functions as F


customers_df = spark.read.parquet(f"{path}/silver/customers.parquet")
normal_join = (
    transactions_100k
    .join(
        customers_df,
        on="customer_id",
        how="left"
    )
)

normal_join.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (17)
+- Project (16)
   +- BroadcastHashJoin LeftOuter BuildRight (15)
      :- Union (11)
      :  :- Scan parquet  (1)
      :  :- Scan parquet  (2)
      :  :- Scan parquet  (3)
      :  :- Scan parquet  (4)
      :  :- Scan parquet  (5)
      :  :- Scan parquet  (6)
      :  :- Scan parquet  (7)
      :  :- Scan parquet  (8)
      :  :- Scan parquet  (9)
      :  +- Scan parquet  (10)
      +- BroadcastExchange (14)
         +- Filter (13)
            +- Scan parquet  (12)


(1) Scan parquet 
Output [14]: [transaction_id#2419, customer_id#2420, card_id#2421, merchant_id#2422, device_id#2423, timestamp#2424, amount#2425, product_type#2426, card_type#2427, payment_type#2428, customer_region#2429, merchant_region#2430, has_identity#2431, is_fraud#2432]
Batched: true
Location: InMemoryFileIndex [file:/c:/Users/u/Desktop/Repositories/Apex_financial/data/gold/fact_transactions.parquet]
ReadSchema: struct<transaction_id:string,customer_id:string,card_

In [46]:
broadcast_join = (
    transactions_100k
    .join(
        F.broadcast(customers_df),
        on="customer_id",
        how="left"
    )
)

broadcast_join.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (17)
+- Project (16)
   +- BroadcastHashJoin LeftOuter BuildRight (15)
      :- Union (11)
      :  :- Scan parquet  (1)
      :  :- Scan parquet  (2)
      :  :- Scan parquet  (3)
      :  :- Scan parquet  (4)
      :  :- Scan parquet  (5)
      :  :- Scan parquet  (6)
      :  :- Scan parquet  (7)
      :  :- Scan parquet  (8)
      :  :- Scan parquet  (9)
      :  +- Scan parquet  (10)
      +- BroadcastExchange (14)
         +- Filter (13)
            +- Scan parquet  (12)


(1) Scan parquet 
Output [14]: [transaction_id#2419, customer_id#2420, card_id#2421, merchant_id#2422, device_id#2423, timestamp#2424, amount#2425, product_type#2426, card_type#2427, payment_type#2428, customer_region#2429, merchant_region#2430, has_identity#2431, is_fraud#2432]
Batched: true
Location: InMemoryFileIndex [file:/c:/Users/u/Desktop/Repositories/Apex_financial/data/gold/fact_transactions.parquet]
ReadSchema: struct<transaction_id:string,customer_id:string,card_

# understanding data skewness

In [50]:
# Create a deliberately skewed dataset

skewed_transactions = (
    transactions_100k
    .withColumn(
        "customer_id",
        F.when(
            F.rand(seed=42) < 0.50,
            F.lit("SKEWED_CUSTOMER")
        ).otherwise(F.col("customer_id"))
    )
)

print("Rows:", skewed_transactions.count())

skewed_transactions.groupBy("customer_id") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(10)

Rows: 100000
+---------------+-----+
|    customer_id|count|
+---------------+-----+
|SKEWED_CUSTOMER|50081|
|        C002326|  135|
|        C000233|  124|
|        C001407|  109|
|        C000688|  108|
|        C001758|  101|
|        C001951|  100|
|        C001436|  100|
|        C000614|   99|
|        C002503|   96|
+---------------+-----+
only showing top 10 rows


In [51]:
skewed_transactions.groupBy("customer_id").count().explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (16)
+- HashAggregate (15)
   +- Exchange (14)
      +- HashAggregate (13)
         +- Project (12)
            +- Union (11)
               :- Scan parquet  (1)
               :- Scan parquet  (2)
               :- Scan parquet  (3)
               :- Scan parquet  (4)
               :- Scan parquet  (5)
               :- Scan parquet  (6)
               :- Scan parquet  (7)
               :- Scan parquet  (8)
               :- Scan parquet  (9)
               +- Scan parquet  (10)


(1) Scan parquet 
Output [1]: [customer_id#2420]
Batched: true
Location: InMemoryFileIndex [file:/c:/Users/u/Desktop/Repositories/Apex_financial/data/gold/fact_transactions.parquet]
ReadSchema: struct<customer_id:string>

(2) Scan parquet 
Output [1]: [customer_id#2478]
Batched: true
Location: InMemoryFileIndex [file:/c:/Users/u/Desktop/Repositories/Apex_financial/data/gold/fact_transactions.parquet]
ReadSchema: struct<customer_id:string>

(3) Scan parquet 
Output [1]:

In [52]:
skewed_customer_summary = (
    skewed_transactions
    .groupBy("customer_id")
    .agg(
        F.count("transaction_id").alias("transaction_count"),
        F.sum("amount").alias("total_amount")
    )
)

skewed_customer_summary.show(10)

+-----------+-----------------+------------------+
|customer_id|transaction_count|      total_amount|
+-----------+-----------------+------------------+
|    C001183|               14|2463.8800000000006|
|    C000729|               31|1089.9900000000002|
|    C001004|               13|            564.26|
|    C001951|              100|          12845.88|
|    C002535|               48|            4602.6|
|    C000388|               95|           9671.43|
|    C001449|               10|            383.88|
|    C000644|               11|            1791.6|
|    C000993|               37|           5273.52|
|    C002107|               22|2961.2999999999997|
+-----------+-----------------+------------------+
only showing top 10 rows


# Scale Testing

In [55]:
transactions_1m = transactions_100k

for _ in range(9):
    transactions_1m = transactions_1m.union(transactions_100k)

print("1M row count:", transactions_1m.count())
print("Partitions:", transactions_1m.rdd.getNumPartitions())

1M row count: 1000000
Partitions: 100


In [56]:
import time

start = time.time()

transactions_1m.groupBy("customer_id").agg(
    F.count("transaction_id").alias("transaction_count"),
    F.sum("amount").alias("total_amount")
).count()

elapsed = time.time() - start

print(f"Execution time: {elapsed:.2f} seconds")

Execution time: 6.85 seconds


In [59]:
scales = {
    "100K": transactions_100k,
    "1M": transactions_1m,
    
}

transactions_10m = transactions_1m

for _ in range(9):
    transactions_10m = transactions_10m.union(transactions_1m)

scales["10M"] = transactions_10m
for name, df in scales.items():
    start = time.time()

    df.groupBy("customer_id").agg(
        F.count("transaction_id").alias("transaction_count"),
        F.sum("amount").alias("total_amount")
    ).count()

    elapsed = time.time() - start

    print(
        f"{name}: "
        f"{df.count():,} rows | "
        f"{df.rdd.getNumPartitions()} partitions | "
        f"{elapsed:.2f}s"
    )

100K: 100,000 rows | 10 partitions | 1.05s
1M: 1,000,000 rows | 100 partitions | 13.19s
10M: 10,000,000 rows | 1000 partitions | 186.67s


In [58]:
for name, df in scales.items():
    start = time.time()

    df.groupBy("customer_id").agg(
        F.count("transaction_id").alias("transaction_count"),
        F.sum("amount").alias("total_amount")
    ).count()

    elapsed = time.time() - start

    print(
        f"{name}: "
        f"{df.count():,} rows | "
        f"{df.rdd.getNumPartitions()} partitions | "
        f"{elapsed:.2f}s"
    )

100K: 100,000 rows | 10 partitions | 1.35s
1M: 1,000,000 rows | 100 partitions | 16.40s
